# Market-Level Transformer Return Timing Model

Based on: Han, Huang, Huang, Zhou, *"Daily Market Return Prediction with Transformer"* (SSRN, this version May 2026).

Unlike `node_transformer_sentiment_forecast.ipynb` (which forecasts individual stocks using a graph + sentiment fusion), this notebook predicts the **next-day aggregate market excess return** using nothing but sequences of the market's own lagged daily returns — no sentiment, no cross-sectional graph. Three models are compared on identical input blocks (past 5, 20, or 60 daily returns):

1. **Transformer** (encoder-only, causal self-attention) — the paper's main model
2. **Random Forest** — baseline
3. **Feedforward Neural Network** — baseline

The paper's central finding: the *raw* ML forecast correctly predicts direction (significant slope in a realized-on-forecast regression) but is badly scaled — a **post-ML regression recalibration** step (re-fit annually) fixes the scale and is what actually delivers usable out-of-sample R² and a tradeable market-timing signal (invest 100% market when the calibrated forecast is positive, else 100% risk-free).

**Where this fits in the app**: this is a *market-wide* beta-exposure signal for **Ledger** (Risk/Portfolio Manager) — see `../tradingskills.md` §6.2. It scales overall market exposure, it does not pick individual tickers, and — per §13 of that doc — it never authorizes auto-execution on its own.

**This notebook reproduces the architecture and methodology, not the paper's exact reported numbers.** Validate independently before trusting any of it with real capital.

In [70]:
pip install torch pandas numpy scikit-learn pandas-datareader


Note: you may need to restart the kernel to use updated packages.


## 0. Requirements

```bash
pip install torch pandas numpy scikit-learn pandas-datareader
```

`pandas-datareader` pulls the Fama-French daily factors (market excess return + risk-free rate) directly from Ken French's data library — the same source ("Professor Fama's website") the paper uses. No API key needed.

In [71]:
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Using device: {DEVICE}")

Using device: mps


## 1. Config

The paper rolls a 5-year window (4 years train + 1 year validation) forward one year at a time across ~90 years of data (1926–2022) — faithfully reproducing that is expensive. `FAST_MODE=True` below restricts to a shorter recent window and a coarser refit cadence so the notebook actually runs end-to-end; set `FAST_MODE=False` and widen `START_DATE` for a fuller replication.

In [72]:
@dataclass
class Config:
    fast_mode: bool = True
    start_date: str = "2000-01-01"   # paper uses 1926-07-01; widen this for a fuller replication
    end_date: str = "2024-12-31"
    block_sizes: tuple = (5, 20, 60)  # lookback lengths compared in the paper
    train_years: int = 4
    val_years: int = 1               # -> 5-year rolling window total, matching the paper
    refit_every_n_years: int = 1      # paper refits every year; raise this under FAST_MODE
    d_model: int = 32
    n_heads: int = 4
    n_layers: int = 2
    dropout: float = 0.1
    lr: float = 1e-4
    epochs: int = 60
    batch_size: int = 32
    rf_n_estimators: int = 250
    rf_max_depth: int = 3

cfg = Config()
if cfg.fast_mode:
    cfg.refit_every_n_years = 3  # only refit every 3 years instead of annually, to keep runtime sane
cfg

Config(fast_mode=True, start_date='2000-01-01', end_date='2024-12-31', block_sizes=(5, 20, 60), train_years=4, val_years=1, refit_every_n_years=3, d_model=32, n_heads=4, n_layers=2, dropout=0.1, lr=0.0001, epochs=60, batch_size=32, rf_n_estimators=250, rf_max_depth=3)

## 2. Data — daily Fama-French market excess return

`Mkt-RF` is the daily aggregate market excess return (in %); dividing by 100 gives the decimal return used as both the model's input sequence and its prediction target.

In [73]:
def load_market_excess_returns(start: str, end: str) -> pd.Series:
    from pandas_datareader import data as pdr

    ff = pdr.DataReader("F-F_Research_Data_Factors_daily", "famafrench", start=start, end=end)[0]
    r = ff["Mkt-RF"] / 100.0
    # pandas_datareader's FamaFrench reader returns a PeriodIndex for daily data;
    # recent pandas versions raise on pd.to_datetime(PeriodIndex) instead of
    # silently converting, so handle it explicitly via .to_timestamp().
    r.index = r.index.to_timestamp() if isinstance(r.index, pd.PeriodIndex) else pd.to_datetime(r.index)
    return r.rename("mkt_excess_return")

## 3. Windowing — sequence input for the Transformer, flattened input for RF/NN

All three models see identical information for a given block size: the past `block_size` daily returns, target = the next day's return. The Transformer keeps the sequence shape (for positional/causal attention); Random Forest and the NN get the same values flattened into a plain feature vector.

In [74]:
def build_windows(returns: pd.Series, block_size: int) -> tuple:
    values = returns.values
    dates = returns.index
    X, y, target_dates = [], [], []
    for i in range(block_size, len(values)):
        X.append(values[i - block_size: i])
        y.append(values[i])
        target_dates.append(dates[i])
    return np.array(X), np.array(y), pd.DatetimeIndex(target_dates)


def rolling_year_slices(target_dates: pd.DatetimeIndex, train_years: int, val_years: int, refit_every: int):
    """Yields (train_idx, val_idx, predict_idx, predict_year) rolling forward one
    predict-year at a time, refitting only every `refit_every` years (paper uses 1).
    """
    years = sorted(target_dates.year.unique())
    window = train_years + val_years
    for i in range(window, len(years), refit_every):
        predict_year = years[i]
        window_years = years[i - window: i]
        train_yrs, val_yrs = window_years[:train_years], window_years[train_years:]

        train_idx = np.where(np.isin(target_dates.year, train_yrs))[0]
        val_idx = np.where(np.isin(target_dates.year, val_yrs))[0]
        predict_idx = np.where(target_dates.year == predict_year)[0]
        if len(predict_idx) == 0:
            continue
        yield train_idx, val_idx, predict_idx, predict_year

## 4. Model — causal Transformer encoder

Matches the paper's description: scalar daily return -> linear embedding (`d_model`) + learned positional encoding -> `n_layers` of masked multi-head self-attention (day *t* attends only to days ≤ *t*) + feed-forward + residual/layer-norm -> take the final timestep's representation -> linear regression head -> single next-day return prediction.

In [75]:
class CausalReturnTransformer(nn.Module):
    def __init__(self, block_size: int, d_model: int, n_heads: int, n_layers: int, dropout: float):
        super().__init__()
        self.input_proj = nn.Linear(1, d_model)
        self.pos_embedding = nn.Parameter(torch.randn(1, block_size, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Linear(d_model, 1)
        # causal mask: True = blocked. Day t may only attend to days <= t.
        mask = torch.triu(torch.ones(block_size, block_size), diagonal=1).bool()
        self.register_buffer("causal_mask", mask)

    def forward(self, x):
        # x: (B, L) daily returns -> (B, L, 1)
        h = self.input_proj(x.unsqueeze(-1)) + self.pos_embedding[:, : x.size(1)]
        h = self.encoder(h, mask=self.causal_mask[: x.size(1), : x.size(1)])
        return self.head(h[:, -1, :]).squeeze(-1)  # prediction as of the last day in the window


def train_transformer(X_train, y_train, X_val, y_val, block_size: int, cfg: Config) -> nn.Module:
    model = CausalReturnTransformer(block_size, cfg.d_model, cfg.n_heads, cfg.n_layers, cfg.dropout).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)

    Xt = torch.tensor(X_train, dtype=torch.float32).to(DEVICE)
    yt = torch.tensor(y_train, dtype=torch.float32).to(DEVICE)
    Xv = torch.tensor(X_val, dtype=torch.float32).to(DEVICE)
    yv = torch.tensor(y_val, dtype=torch.float32).to(DEVICE)

    best_val, best_state = float("inf"), None
    n = len(Xt)
    for epoch in range(cfg.epochs):
        model.train()
        perm = torch.randperm(n)
        for start in range(0, n, cfg.batch_size):
            idx = perm[start: start + cfg.batch_size]
            optimizer.zero_grad()
            pred = model(Xt[idx])
            loss = F.mse_loss(pred, yt[idx])
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = F.mse_loss(model(Xv), yv).item()
        if val_loss < best_val:
            best_val, best_state = val_loss, {k: v.clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    model.eval()
    return model


def predict_transformer(model: nn.Module, X: np.ndarray) -> np.ndarray:
    with torch.no_grad():
        return model(torch.tensor(X, dtype=torch.float32).to(DEVICE)).cpu().numpy()

## 5. Baselines — Random Forest and feedforward Neural Network

Same flattened lagged-return features, same rolling windows — only the model class differs, isolating what the Transformer's sequence-awareness actually buys you.

In [76]:
def train_random_forest(X_train, y_train, cfg: Config) -> RandomForestRegressor:
    model = RandomForestRegressor(
        n_estimators=cfg.rf_n_estimators, max_depth=cfg.rf_max_depth, random_state=42, n_jobs=-1,
    )
    model.fit(X_train, y_train)
    return model


def train_neural_network(X_train, y_train) -> MLPRegressor:
    # Geometric-pyramid-ish sizing per the paper's description; sklearn's MLP
    # is a reasonable stand-in for the paper's own feedforward net.
    model = MLPRegressor(
        hidden_layer_sizes=(64, 32, 16, 8), activation="relu", solver="adam",
        learning_rate_init=0.001, max_iter=500, random_state=42,
    )
    model.fit(X_train, y_train)
    return model


def naive_average_benchmark(X: np.ndarray) -> np.ndarray:
    """The paper's other benchmark: the simple average of the past window's returns."""
    return X.mean(axis=1)

## 6. Rolling backtest — generate out-of-sample forecasts for every model

For each block size, walks forward year by year: fit on the rolling window's train years, pick/validate on the validation year (kept simple here — see note below), predict the following year, then advance.

In [77]:
def run_rolling_backtest(returns: pd.Series, block_size: int, cfg: Config) -> tuple:
    """Returns (predictions_df, last_transformer). `last_transformer` is the
    model trained on the *final* rolling window — i.e. the most recent one,
    which is what should actually serve live inference. Every earlier
    iteration's model is intentionally discarded once its predictions are
    recorded; only the last one is production-relevant.
    """
    X, y, target_dates = build_windows(returns, block_size)
    records = []
    last_transformer = None

    for train_idx, val_idx, predict_idx, predict_year in rolling_year_slices(
        target_dates, cfg.train_years, cfg.val_years, cfg.refit_every_n_years
    ):
        X_train, y_train = X[train_idx], y[train_idx]
        X_val, y_val = X[val_idx], y[val_idx]
        X_pred, y_pred, dates_pred = X[predict_idx], y[predict_idx], target_dates[predict_idx]

        transformer = train_transformer(X_train, y_train, X_val, y_val, block_size, cfg)
        rf = train_random_forest(X_train, y_train, cfg)
        nn_model = train_neural_network(X_train, y_train)
        last_transformer = transformer  # kept after the loop ends -> the most recently trained model

        for date, realized, x_row in zip(dates_pred, y_pred, X_pred):
            records.append({
                "date": date,
                "block_size": block_size,
                "realized_return": realized,
                "transformer_forecast": float(predict_transformer(transformer, x_row[None, :])[0]),
                "rf_forecast": float(rf.predict(x_row[None, :])[0]),
                "nn_forecast": float(nn_model.predict(x_row[None, :])[0]),
                "naive_avg_forecast": float(x_row.mean()),
            })

    return pd.DataFrame.from_records(records), last_transformer


# results = {bs: run_rolling_backtest(returns, bs, cfg) for bs in cfg.block_sizes}

## 7. Post-ML regression recalibration

The paper's key step. Each year, regress *realized* returns on the raw forecast using an expanding window of all prior out-of-sample (forecast, realized) pairs, then apply that year's regression to rescale the current year's raw forecast. This fixes the scale mismatch (raw slope < 1) without needing to touch the underlying model.

In [78]:
from sklearn.linear_model import LinearRegression


def post_ml_recalibrate(df: pd.DataFrame, forecast_col: str) -> pd.Series:
    """Expanding-window OLS: realized_return ~ forecast, refit each year on prior years only."""
    df = df.sort_values("date").reset_index(drop=True)
    years = df["date"].dt.year
    calibrated = pd.Series(index=df.index, dtype=float)

    for year in sorted(years.unique()):
        prior = df[years < year]
        current = years == year
        if len(prior) < 30:  # not enough history yet to fit a stable regression
            calibrated.loc[current] = df.loc[current, forecast_col]
            continue
        reg = LinearRegression().fit(prior[[forecast_col]], prior["realized_return"])
        calibrated.loc[current] = reg.predict(df.loc[current, [forecast_col]])

    return calibrated


def fit_final_calibration(df: pd.DataFrame, forecast_col: str) -> LinearRegression:
    """Fits the calibration regression on *all* available (forecast, realized)
    pairs — the one that should actually accompany the exported model for
    live inference today, as opposed to post_ml_recalibrate's year-by-year
    walk-forward version used only for backtesting.
    """
    return LinearRegression().fit(df[[forecast_col]], df["realized_return"])

In [79]:
pip install statsmodels

Note: you may need to restart the kernel to use updated packages.


## 8. Evaluation — predictive regression, out-of-sample R², market timing

Mirrors the paper's three checks: (1) does the forecast significantly predict next-day returns (Mincer-Zarnowitz-style slope + t-stat), (2) out-of-sample R² (Campbell-Thompson, eq. 6 in the paper) vs. an expanding historical average, (3) a market-timing Sharpe ratio.

In [80]:
import statsmodels.api as sm


def predictive_regression(df: pd.DataFrame, forecast_col: str) -> dict:
    X = sm.add_constant(df[forecast_col])
    ols = sm.OLS(df["realized_return"], X).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    return {"slope": ols.params[forecast_col], "t_stat": ols.tvalues[forecast_col], "r2": ols.rsquared}


def out_of_sample_r2(df: pd.DataFrame, forecast_col: str) -> float:
    df = df.sort_values("date")
    expanding_avg = df["realized_return"].expanding().mean().shift(1)  # avg return using only prior days
    valid = expanding_avg.notna()
    r, rhat, rbar = df.loc[valid, "realized_return"], df.loc[valid, forecast_col], expanding_avg[valid]
    return 1 - ((r - rhat) ** 2).sum() / ((r - rbar) ** 2).sum()


def market_timing_backtest(df: pd.DataFrame, forecast_col: str, risk_free_daily: float = 0.0) -> dict:
    """Invest 100% market when forecast > 0, else 100% risk-free."""
    df = df.sort_values("date")
    strategy_return = np.where(df[forecast_col] > 0, df["realized_return"], risk_free_daily)
    ann_return = strategy_return.mean() * 252
    ann_vol = strategy_return.std() * np.sqrt(252)
    sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan

    buy_hold_return = df["realized_return"].mean() * 252
    buy_hold_vol = df["realized_return"].std() * np.sqrt(252)
    buy_hold_sharpe = buy_hold_return / buy_hold_vol if buy_hold_vol > 0 else np.nan

    success = np.where(
        df["realized_return"] > 0, strategy_return > 0, strategy_return >= risk_free_daily
    ).mean()

    return {
        "strategy_ann_return": ann_return, "strategy_ann_vol": ann_vol, "strategy_sharpe": sharpe,
        "buy_hold_ann_return": buy_hold_return, "buy_hold_sharpe": buy_hold_sharpe,
        "success_rate": success,
    }

## 9. Run end-to-end

Uncomment to actually pull data and run the full rolling backtest for every block size and model. Left commented by default since the rolling refit loop trains multiple models per iteration and can take a while even in `FAST_MODE`.

In [81]:
returns = load_market_excess_returns(cfg.start_date, cfg.end_date)

summary = []
trained_models = {}      # block_size -> CausalReturnTransformer (most recent rolling window)
final_calibrations = {}  # block_size -> LinearRegression (fit on all history, for live inference)

for block_size in cfg.block_sizes:
    df, last_model = run_rolling_backtest(returns, block_size, cfg)
    df["transformer_calibrated"] = post_ml_recalibrate(df, "transformer_forecast")
    df["rf_calibrated"] = post_ml_recalibrate(df, "rf_forecast")
    df["nn_calibrated"] = post_ml_recalibrate(df, "nn_forecast")

    trained_models[block_size] = last_model
    final_calibrations[block_size] = fit_final_calibration(df, "transformer_forecast")

    for forecast_col in ["transformer_forecast", "transformer_calibrated", "naive_avg_forecast"]:
        reg = predictive_regression(df, forecast_col)
        r2 = out_of_sample_r2(df, forecast_col)
        timing = market_timing_backtest(df, forecast_col)
        summary.append({"block_size": block_size, "forecast": forecast_col, **reg, "oos_r2": r2, **timing})

pd.DataFrame(summary)

,block_size,forecast,slope,t_stat,r2,oos_r2,strategy_ann_return,strategy_ann_vol,strategy_sharpe,buy_hold_ann_return,buy_hold_sharpe,success_rate
0,5,transformer_forecast,0.746843,1.773246,0.008541,0.009249,0.062721,0.198426,0.316094,0.06588,0.279859,0.541690
1,5,transformer_calibrated,0.355262,0.549791,0.000636,-0.000251,0.004531,0.162359,0.027908,0.06588,0.279859,0.480431
2,5,naive_avg_forecast,-0.256557,-2.068096,0.009827,-0.222984,-0.029888,0.141595,-0.211084,0.06588,0.279859,0.496313
3,20,transformer_forecast,-0.031208,-0.128671,0.000040,-0.040905,0.033033,0.166789,0.198053,0.06588,0.279859,0.520136
4,20,transformer_calibrated,-0.968047,-2.427307,0.009899,-0.030317,-0.064479,0.134074,-0.480924,0.06588,0.279859,0.469654
5,20,naive_avg_forecast,-0.197985,-0.860973,0.001384,-0.046720,0.077344,0.116115,0.666096,0.06588,0.279859,0.536018
6,60,transformer_forecast,-0.394130,-0.839634,0.004245,-0.047062,0.022027,0.192000,0.114723,0.06588,0.279859,0.521271
7,60,transformer_calibrated,-0.548940,-0.815487,0.002776,-0.016967,-0.055017,0.154381,-0.356372,0.06588,0.279859,0.477028
8,60,naive_avg_forecast,-0.418603,-1.130502,0.001851,-0.016909,0.087292,0.111491,0.782954,0.06588,0.279859,0.551900


## 10. Export for the Python service

In [82]:
import json
from pathlib import Path

ARTIFACT_DIR = Path("artifacts_market_timing")


def export_model(model: CausalReturnTransformer, block_size: int, calibration_reg: LinearRegression,
                  cfg: Config, out_dir: Path = ARTIFACT_DIR):
    out_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), out_dir / f"transformer_block{block_size}.pt")
    with open(out_dir / f"calibration_block{block_size}.json", "w") as f:
        json.dump({"coef": float(calibration_reg.coef_[0]), "intercept": float(calibration_reg.intercept_)}, f)
    with open(out_dir / "config.json", "w") as f:
        json.dump({"d_model": cfg.d_model, "n_heads": cfg.n_heads, "n_layers": cfg.n_layers,
                    "dropout": cfg.dropout, "block_sizes": list(cfg.block_sizes)}, f, indent=2)
    print(f"Saved artifacts to {out_dir.resolve()}")


def predict_market_timing_signal(recent_returns: np.ndarray, block_size: int,
                                  artifact_dir: Path = ARTIFACT_DIR) -> dict:
    """recent_returns: the last `block_size` daily market excess returns. This is the
    function Ledger calls to get the current exposure signal for the report card/§10.
    """
    with open(artifact_dir / "config.json") as f:
        saved_cfg = json.load(f)
    model = CausalReturnTransformer(block_size, saved_cfg["d_model"], saved_cfg["n_heads"],
                                     saved_cfg["n_layers"], saved_cfg["dropout"]).to(DEVICE)
    model.load_state_dict(torch.load(artifact_dir / f"transformer_block{block_size}.pt", map_location=DEVICE))
    model.eval()

    with open(artifact_dir / f"calibration_block{block_size}.json") as f:
        calib = json.load(f)

    raw = float(predict_transformer(model, recent_returns[None, :])[0])
    calibrated = calib["coef"] * raw + calib["intercept"]
    return {
        "raw_forecast": raw,
        "calibrated_forecast": calibrated,
        "timing_signal": "long_market" if calibrated > 0 else "risk_free",
    }


for block_size in cfg.block_sizes:
    export_model(trained_models[block_size], block_size, final_calibrations[block_size], cfg)

Saved artifacts to /Users/spanda/IdeaProjects/tradingapp/notebooks/artifacts_market_timing
Saved artifacts to /Users/spanda/IdeaProjects/tradingapp/notebooks/artifacts_market_timing
Saved artifacts to /Users/spanda/IdeaProjects/tradingapp/notebooks/artifacts_market_timing


## 11. Caveats & integration notes

- **Market-wide overlay, not a stock pick.** This model answers "how much overall market exposure should the book carry today," feeding Ledger's position sizing (§5/§6.2 in `tradingskills.md`). It is entirely separate from Cortex's per-ticker forecast (§6.1) and from Alpha's report card (§10).
- **The rolling refit scheme is the point, not an implementation detail.** The paper explicitly attributes predictive power to the combination of the nonlinear architecture *and* the yearly re-estimation, not to the attention mechanism selectively weighting specific lagged days (its attention weights come out fairly flat). Skipping the rolling refit and training one static model on all history would not reproduce the effect described.
- **Recalibrate, don't just trust the raw forecast.** The post-ML regression step in §7 above is not optional polish — the raw forecast's scale is unreliable on its own per the paper's own analysis.
- **`FAST_MODE` trades fidelity for runtime.** Widen `start_date` and set `refit_every_n_years=1` for a fuller replication; expect a much longer run.
- **Validate before relying on it.** Reported Sharpe ratios (~1.1–1.3) and out-of-sample R² (~1%) are as published; treat them as a starting hypothesis to test on your own data, not a guarantee.
- **Nothing here authorizes auto-execution** — per §13 of `tradingskills.md`, any resulting exposure change goes through the human-confirmation step before an order is placed.